In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.4 MB/s eta 0:00:00


In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import os
import json
import pandas as pd
from torch import float16
from tqdm import tqdm
import torch, gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


class DocumentIndexer:
    def __init__(self, model_name='Qwen/Qwen3-Embedding-0.6B'):
        self.model = SentenceTransformer(
            model_name,
            model_kwargs={
                #"attn_implementation": "flash_attention_2",
                "device_map": "auto",
                "torch_dtype": float16,
            },
            tokenizer_kwargs={"padding_side": "left"},
        )
        self.doc_index = None
        self.doc_texts = []
        self.doc_ids = []
        self.doc_embeddings = None       # np.ndarray, shape (N, d), float32, L2-normalised
        self.sentence_texts = {}
        self.sentence_indexes = {}
        self.d = None

    # ------------------------------------------------------------------
    # helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _weighted_mean_embedding(sentence_embeddings: np.ndarray,
                                  sentences: list[str]) -> np.ndarray:
        """
        Weighted mean of sentence embeddings where the weight of each
        sentence is proportional to its character length.
        Returns an L2-normalised vector.
        """
        weights = np.array([max(len(s), 1) for s in sentences], dtype=np.float32)
        weights /= weights.sum()                          # normalise weights → sum to 1
        doc_emb = (sentence_embeddings * weights[:, None]).sum(axis=0)  # weighted sum
        doc_emb = doc_emb.astype(np.float32)
        norm = np.linalg.norm(doc_emb)
        if norm > 0:
            doc_emb /= norm
        return doc_emb

    @staticmethod
    def _l2_normalise(matrix: np.ndarray) -> np.ndarray:
        """Row-wise L2 normalisation."""
        matrix = matrix.astype(np.float32)
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)         # avoid div-by-zero
        return matrix / norms

    # ------------------------------------------------------------------
    # CSV-based ingestion
    # ------------------------------------------------------------------

    def add_documents_from_df(self,
                               df: pd.DataFrame,
                               text_col: str = "content",
                               id_col: str = "res_id2") -> None:
        """
        Encode a batch of documents supplied as a DataFrame.

        Parameters
        ----------
        df        : DataFrame with at least `id_col` and `text_col` columns.
        text_col  : Column that holds the raw document text.
        id_col    : Column that holds the unique document identifier.
        """
        batch_doc_embeddings = []

        for local_idx, row in tqdm(df.iterrows(), total=len(df), desc="Encoding docs"):
            doc_id   = row[id_col]
            doc_text = str(row[text_col])

            self.doc_ids.append(doc_id)
            self.doc_texts.append(doc_text)

            # --- sentence-level ---
            sentences = [s.strip() for s in doc_text.split(". ") if s.strip()]
            if not sentences:
                sentences = [doc_text]

            self.sentence_texts[doc_id] = sentences

            sent_embs = self.model.encode(
                sentences,
                convert_to_numpy=True,
                batch_size=8,
                normalize_embeddings=False,   # we normalise manually below
            ).astype(np.float32)

            # L2-normalise sentence embeddings before storing / adding to FAISS
            sent_embs_normed = self._l2_normalise(sent_embs)

            torch.cuda.empty_cache()
            gc.collect()

            # sentence-level FAISS index (inner product == cosine after L2 norm)
            d = sent_embs_normed.shape[1]
            sent_index = faiss.IndexFlatIP(d)
            sent_index.add(sent_embs_normed)
            self.sentence_indexes[doc_id] = sent_index

            # doc embedding: length-weighted mean, then L2-normalised
            doc_emb = self._weighted_mean_embedding(sent_embs_normed, sentences)
            batch_doc_embeddings.append(doc_emb)

        # --- build / extend doc-level FAISS index ---
        batch_matrix = np.vstack(batch_doc_embeddings).astype(np.float32)
        # batch_matrix rows are already L2-normalised individually, but
        # stack can introduce tiny float drift — re-normalise to be safe
        batch_matrix = self._l2_normalise(batch_matrix)

        self.d = batch_matrix.shape[1]
        self.doc_embeddings = batch_matrix

        self.doc_index = faiss.IndexFlatIP(self.d)
        self.doc_index.add(batch_matrix)

        print(f"Indexed {len(df)} documents  |  FAISS total: {self.doc_index.ntotal}")

    # ------------------------------------------------------------------
    # diagnostics
    # ------------------------------------------------------------------

    def print_embedding_summary(self, local_idx: int = 0) -> None:
        """Print shape and norm of the doc embedding at `local_idx`."""
        if self.doc_embeddings is None or len(self.doc_embeddings) == 0:
            print("No embeddings available yet.")
            return
        emb = self.doc_embeddings[local_idx]
        print(f"[summary] doc[{local_idx}]  shape={emb.shape}  "
              f"norm={np.linalg.norm(emb):.6f}  dtype={emb.dtype}")

    def verify_index(self, doc_id: int = 0) -> None:
        """
        Query the doc index with its own embedding and assert it returns
        itself as the top hit (sanity-check that IP == cosine after norm).
        """
        if self.doc_index is None:
            print("Index not built yet.")
            return
        query = self.doc_embeddings[doc_id:doc_id + 1]
        D, I = self.doc_index.search(query, k=1)
        ok = I[0][0] == doc_id
        print(f"[verify] doc[{doc_id}] → top-1 idx={I[0][0]}  "
              f"score={D[0][0]:.6f}  {'✓ PASS' if ok else '✗ FAIL'}")

    # ------------------------------------------------------------------
    # persistence
    # ------------------------------------------------------------------

    def save_index(self,
                   index_path: str = "faiss_docs.bin",
                   metadata_path: str = "faiss_docs.npy") -> None:
        faiss.write_index(self.doc_index, index_path)
        metadata = [
            {"doc_id": doc_id, "content": doc_text}
            for doc_id, doc_text in zip(self.doc_ids, self.doc_texts)
        ]
        np.save(metadata_path, np.array(metadata, dtype=object))
        print(f"Index → {index_path}  |  Metadata → {metadata_path}")

In [5]:
import faiss
import numpy as np
import pandas as pd
import os

# ----------------------------
# SETTINGS
# ----------------------------
INPUT_CSV  = '/content/drive/MyDrive/ga_resolutions_1946_2019.csv'
BASE_DIR   = '/content/drive/MyDrive/'
BATCH_SIZE = 10000

INDEX_PATH    = BASE_DIR + "faiss_docs_resolutions.bin"
METADATA_PATH = BASE_DIR + "faiss_docs_resolutions.npy"

# ----------------------------
# LOAD DATA
# ----------------------------
df_full = pd.read_csv(INPUT_CSV)
df_full = df_full[["res_id2", "content"]].drop_duplicates().reset_index(drop=True)
print(f"Total unique resolutions: {len(df_full)}")

# ----------------------------
# CHECKPOINT: read already-processed IDs from .npy
# ----------------------------
if os.path.exists(METADATA_PATH):
    existing_metadata = np.load(METADATA_PATH, allow_pickle=True)
    done_ids = set(item["doc_id"] for item in existing_metadata)
    print(f"Already indexed: {len(done_ids)} — skipping these.")
else:
    done_ids = set()

df_todo = df_full[~df_full["res_id2"].isin(done_ids)].reset_index(drop=True)
print(f"Remaining to index: {len(df_todo)}")

df_batch = df_todo.iloc[:BATCH_SIZE].reset_index(drop=True)
print(f"This run: {len(df_batch)} resolutions (batch_size={BATCH_SIZE})")

if df_batch.empty:
    print("Nothing to do — all resolutions already indexed.")
else:
    # ----------------------------
    # INDEX BATCH
    # ----------------------------
    indexer = DocumentIndexer()
    indexer.add_documents_from_df(df_batch, text_col="content", id_col="res_id2")

    # Quick sanity checks on the batch
    #indexer.print_embedding_summary(11)
    #indexer.verify_index(doc_id=11)

    # ----------------------------
    # MERGE WITH EXISTING INDEX (if any) AND SAVE
    # ----------------------------
    if os.path.exists(INDEX_PATH) and os.path.exists(METADATA_PATH):
        print("Merging with existing index...")

        existing_index    = faiss.read_index(INDEX_PATH)
        existing_metadata = np.load(METADATA_PATH, allow_pickle=True).tolist()

        # Merge vectors: add new batch vectors into the existing index
        existing_index.add(indexer.doc_embeddings)

        # Merge metadata lists
        new_metadata = [
            {"doc_id": doc_id, "content": doc_text}
            for doc_id, doc_text in zip(indexer.doc_ids, indexer.doc_texts)
        ]
        merged_metadata = existing_metadata + new_metadata

        faiss.write_index(existing_index, INDEX_PATH)
        np.save(METADATA_PATH, np.array(merged_metadata, dtype=object))
        print(f"Merged index now contains {existing_index.ntotal} documents.")

    else:
        # First run — just save normally
        indexer.save_index(index_path=INDEX_PATH, metadata_path=METADATA_PATH)

    print(f"\nBatch done. Rerun to process the next {BATCH_SIZE}.")

Total unique resolutions: 17913


Already indexed: 11010 — skipping these.
Remaining to index: 6903
This run: 6903 resolutions (batch_size=10000)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Encoding docs: 100%|██████████| 6903/6903 [1:41:08<00:00,  1.14it/s]


Indexed 6903 documents  |  FAISS total: 6903
Merging with existing index...
Merged index now contains 17913 documents.

Batch done. Rerun to process the next 10000.
